# Phase 5: Next-Day Cell-Density Baselines

## 1. Problem definition
Estimate `Target_Cell_Density_Next_Day` from observations available at the current day. The 689 samples are repeated measurements from 61 `Exp` + `Funnel#` cultures across four experiments; they are not independent experimental units. No random row split, deep-learning model, tuning, or deployment claim is used here.

## 2. Dataset loading

In [ ]:
from pathlib import Path
import importlib
import importlib.util
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

INPUT_PATH = Path("data/processed/hiltunen_next_day_density_basic.csv")
search_locations = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
PROJECT_ROOT = next((path for path in search_locations if (path / INPUT_PATH).is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"Could not find {INPUT_PATH} from this directory or its parents.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.models.baseline_models as baseline_models

baseline_models = importlib.reload(baseline_models)
datasets = baseline_models.load_candidate_datasets(PROJECT_ROOT)
basic_data = datasets["basic"]
ph_data = datasets["ph"]
xgboost_available = importlib.util.find_spec("xgboost") is not None
print("Basic candidate:", basic_data.shape, basic_data.columns.tolist())
print("pH candidate:", ph_data.shape, ph_data.columns.tolist())
print("XGBoost already installed:", xgboost_available)
print("No input CSV is modified by this notebook.")

## 3. Feature and target definition

In [ ]:
feature_definitions = pd.DataFrame([
    {"Feature_Set": "basic", "Predictors": baseline_models.FEATURE_SETS["basic"]["features"]},
    {"Feature_Set": "ph", "Predictors": baseline_models.FEATURE_SETS["ph"]["features"]},
])
display(feature_definitions)
print("Target:", baseline_models.TARGET_COLUMN)
print("Grouping identifiers excluded from predictors:", baseline_models.IDENTIFIER_COLUMNS)
for feature_set, frame in datasets.items():
    predictors = baseline_models.FEATURE_SETS[feature_set]["features"]
    assert not set(baseline_models.IDENTIFIER_COLUMNS) & set(predictors)
    assert baseline_models.TARGET_COLUMN not in predictors
    assert frame[baseline_models.TARGET_COLUMN].notna().all()
print("Both input schemas are complete; no imputation is required.")

## 4. Exploratory checks

In [ ]:
sns.set_theme(style="whitegrid", context="notebook")
day_counts = basic_data.groupby("Current_Day").size().rename("Valid samples").reset_index()
display(day_counts)
for column in ["Medium", "CO2", "Light"]:
    print(f"{column} categories:", sorted(basic_data[column].unique().tolist()))

fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
sns.histplot(data=basic_data, x="Target_Cell_Density_Next_Day", bins=30, ax=axes[0, 0], color="#287a78")
axes[0, 0].set(title="Next-day cell-density target", xlabel="Cell density (10^6 cells/mL)", ylabel="Samples")
sns.histplot(data=basic_data, x="Current_Cell_Density", bins=30, ax=axes[0, 1], color="#b45f45")
axes[0, 1].set(title="Current-day cell density", xlabel="Cell density (10^6 cells/mL)", ylabel="Samples")
sns.histplot(data=ph_data, x="Current_pH", bins=25, ax=axes[1, 0], color="#5577a8")
axes[1, 0].set(title="Current-day pH", xlabel="pH", ylabel="Samples")
sns.barplot(data=day_counts, x="Current_Day", y="Valid samples", color="#7c8450", ax=axes[1, 1])
axes[1, 1].set(title="Eligible next-day samples by current day", xlabel="Current day", ylabel="Samples")
plt.show()

numeric_columns = ["Current_Day", "Current_Cell_Density", "Target_Cell_Density_Next_Day"]
print("Spearman correlations (basic numeric variables; descriptive only):")
display(basic_data[numeric_columns].corr(method="spearman"))
print("Spearman correlations including current pH (descriptive, not causal or predictive evidence):")
display(ph_data[["Current_Day", "Current_Cell_Density", "Current_pH", "Target_Cell_Density_Next_Day"]].corr(method="spearman"))

## 5. Persistence baseline
For every held-out fold, persistence predicts tomorrow's density as today's observed density. It is scored on exactly the same held-out rows as each ML model and is reported under both candidate feature labels.

## 6. Linear Regression
Numeric scaling and one-hot encoding are fit inside a scikit-learn `Pipeline` separately within each training fold. Categories unseen during training are ignored by the encoder rather than causing validation failure.

## 7. Random Forest
The baseline uses 100 trees and `random_state=42`; no extensive tuning is performed. Encoding is fit inside each fold's pipeline.

## 8. XGBoost
XGBoost is evaluated only when it is already installed. In this environment it is not installed, so it is omitted; no package is installed automatically.

## 9. Grouped cross-validation by culture
Five-fold `GroupKFold` keeps every `Exp` + `Funnel#` culture wholly within either training or validation for each fold. This estimates performance on cultures not seen during fitting, while experiments and treatment context may still be shared.

## 10. Leave-one-experiment-out evaluation
Each of the four experiments is held out once. This is an exploratory estimate of transfer to a new experiment and is highly data-limited because only four independent experiment groups exist.

In [ ]:
results, fold_results, evaluation_metadata = baseline_models.run_baseline_evaluation(PROJECT_ROOT)
print("\nExperiment-held-out sample counts:")
display(
    fold_results.loc[fold_results["Evaluation_Protocol"].eq("LeaveOneExperimentOut"),
                 ["Feature_Set", "Model", "Fold", "N_Train", "N_Validation", "MAE", "RMSE", "R2"]]
)

## 11. Results tables

In [ ]:
comparison_columns = [
    "Feature_Set", "Model", "Evaluation_Protocol",
    "MAE", "MAE_Std", "RMSE", "RMSE_Std", "R2", "R2_Std",
    "MAE_Delta_vs_Persistence", "RMSE_Delta_vs_Persistence", "R2_Delta_vs_Persistence",
]
display(results[comparison_columns].round(4))
print("Detailed fold-level rows:", fold_results.shape)
display(fold_results.head(15).round(4))
print("Summary means/stds are unweighted across folds; pooled out-of-fold metrics are also saved in the summary CSV.")

## 12. Interpretation

In [ ]:
ph_effect = results.pivot(index=["Model", "Evaluation_Protocol"], columns="Feature_Set", values=["MAE", "RMSE", "R2"])
ph_differences = pd.DataFrame(index=ph_effect.index)
for metric in ["MAE", "RMSE", "R2"]:
    ph_differences[f"{metric}_PH_minus_basic"] = ph_effect[(metric, "ph")] - ph_effect[(metric, "basic")]
display(ph_differences.round(4))
print("Negative MAE/RMSE differences favor the pH feature set; positive R2 differences favor it.")
print("Assess these changes alongside fold variation and the four-experiment limitation; they are not proof of pH benefit.")

## 13. Limitations
The 689 samples are repeated pairs from 61 cultures and four experiments. Culture-group CV prevents within-culture overlap but can share experiment context; leave-one-experiment-out has only four folds and varies in held-out sample size. Experiment-held-out results vary substantially, so the strong culture-fold Random Forest results should not be read as robust transfer. Candidate data contains no missing feature/target values, so no imputation was needed. Results concern experimental Chlorella conditions only, not Liquid Tree hardware or sensor deployment. No model artifact is saved.

## 14. Recommendation for the next modelling phase
The evidence supports proceeding only with carefully qualified baseline analysis, not directly to ANN/LSTM/GRU. First inspect the fold-level experiment transfer failures and define the intended deployment estimand; collect additional independent experiments and validate against Liquid Tree sensor data before considering higher-capacity models. Treat the current results as proof-of-concept model development, not deployment readiness.